In [ ]:
import random
from queue import PriorityQueue

def fitness(ind):
    N = len(ind)
    non_attacking = N*(N-1) // 2
    attacking = 0
    for i in range(N):
        for j in range(i+1, N):
            if abs(ind[i] - ind[j]) == abs(i - j):
                attacking += 1
    
    return non_attacking - attacking

def init_population(size, N):
    population = []
    node = list(range(N))
    for _ in range(size):
        ind = node[:]
        random.shuffle(ind)
        population.append(ind)
    return population

def tournament_selection(pop, k):
    selected = random.sample(pop, k)
    return max(selected, key=fitness)

def roulette_selection(pop):
    fit = [1/fitness(ind) for ind in pop]

    total_fitness = sum(fit)

    probs = [ind/total_fitness for ind in fit]
    parents = random.choices(pop, weights=probs, k=2)

    return parents[0], parents[1]

def crossover_cycle(p1, p2):
    size = len(p1)
    child = [None] * size
    cycles = [False] * size
    index = 0

    while not all(cycles):

        while cycles[index]:
            index += 1
        start = index

        while True:
            child[index] = p1[index]
            cycles[index] = True
            val = p2[index]
            index = p1.index(val)
            if index == start:
                break
        
    for i in range(size):
        if child[i] == None:
            child[i] = p2[i]
    
    return child

def pmx(p1, p2):
    size = len(p1)
    c1 = [None] * size
    c2 = [None] * size

    a = random.randint(0, size - 2)
    b = random.randint(a + 1, size - 1)

    c1[a:b+1] = p1[a:b+1]
    c2[a:b+1] = p2[a:b+1]

    def fill(p, c):
        for i in range(size):
            if c[i] == None:
                gen = p[i]
                while gen in c:
                    gen = p[c.index(gen)]
                c[i] = gen
        return c
    
    c1 = fill(p2, c1)
    c2 = fill(p1, c2)

    return c1, c2
    
def inversion_mutation(ind, rate=0.1):
    if random.random() < rate:
        i,j = sorted(random.sample(range(len(ind)), 2))
        ind[i:j+1] = reversed(ind[i:j+1])
    return ind

def swapping_mutation(ind, rate=0.1):
    for i in range(len(ind)):
        if random.random() < rate:
            j = random.randint(0, len(ind) - 2)
            ind[i], ind[j] = ind[j], ind[i]
    return ind

def genetic_algo(pop_size, generations, N):
    population = init_population(pop_size, N)
    best = max(population, key=fitness)
    best_fitness = fitness(best)

    for gen in range(generations):
        new_pop = []

        while len(new_pop) < pop_size:
            p1 = tournament_selection(population, 1)
            p2 = tournament_selection(population, 1)
            c1 = crossover_cycle(p1, p2)
            c2 = crossover_cycle(p2, p1)
            c1 = inversion_mutation(c1, 0.2)
            c2 = inversion_mutation(c2, 0.2)
            new_pop.extend([c1, c2])
        population = new_pop[:pop_size]

        curr = max(population, key=fitness)
        if fitness(curr) > best_fitness:
            best = curr
            best_fitness = fitness(curr)
        
        if best_fitness == N*(N-1) // 2:
            break
    
    return best, best_fitness

best, best_fitness = genetic_algo(100, 500, 8)
print(best)
print(best_fitness)

[4, 2, 7, 3, 6, 0, 5, 1]
28
